In [1]:
import polars as pl
from huggingface_hub import hf_hub_download

In [2]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

In [3]:
local_path = hf_hub_download(
    repo_id="McAuley-Lab/Amazon-Reviews-2023",
    filename="raw/review_categories/Beauty_and_Personal_Care.jsonl",
    repo_type="dataset",
    token=HF_TOKEN
)

raw/review_categories/Beauty_and_Persona(…):   0%|          | 0.00/11.0G [00:00<?, ?B/s]

In [4]:
local_path

'/root/.cache/huggingface/hub/datasets--McAuley-Lab--Amazon-Reviews-2023/snapshots/2b6d039ed471f2ba5fd2acb718bf33b0a7e5598e/raw/review_categories/Beauty_and_Personal_Care.jsonl'

In [7]:
!ls -lhL $local_path

-rw-r--r-- 1 root root 11G Jun 15 20:01 /root/.cache/huggingface/hub/datasets--McAuley-Lab--Amazon-Reviews-2023/snapshots/2b6d039ed471f2ba5fd2acb718bf33b0a7e5598e/raw/review_categories/Beauty_and_Personal_Care.jsonl


In [42]:
df_head = pl.read_ndjson(local_path, n_rows=5)
df_head.columns

['rating',
 'title',
 'text',
 'images',
 'asin',
 'parent_asin',
 'user_id',
 'timestamp',
 'helpful_vote',
 'verified_purchase']

In [ ]:
lazy_df = pl.scan_ndjson(local_path)

In [64]:
df = (
    lazy_df
    .select(["rating", "asin", "user_id", "timestamp"])
    .collect()
).filter(pl.col('rating') >= 4)

In [65]:
df.height

18079660

In [66]:
start, end = df['timestamp'].min(),  df['timestamp'].max()
start, end

(954576629000, 1694565034585)

In [69]:
split = [
  946684800000,
  978307200000,
  1009843200000,
  1041379200000,
  1072915200000,
  1104537600000,
  1136073600000,
  1167609600000,
  1199145600000,
  1230768000000,
  1262304000000,
  1293840000000,
  1325376000000,
  1356998400000,
  1388534400000,
  1420070400000,
  1451606400000,
  1483228800000,
  1514764800000,
  1546300800000,
  1577836800000,
  1609459200000,
  1640995200000,
  1672531200000
]

for start, end in zip(split[0:-1], split[1:]):
    print(df.filter((pl.col('timestamp') >= start) & (pl.col('timestamp') < end)).height)

15
41
90
240
689
1833
2686
7576
11359
17088
31507
67855
139197
402953
713778
1171860
1399164
1379841
1452869
2035908
2705598
2697405
2620530


In [83]:
df1 = df.filter((pl.col('timestamp') >= 1609459200000) & (pl.col('timestamp') < 1672531200000))

In [84]:
count1 = df1.group_by('asin').agg(pl.len()).select('asin', 'len')
count1.describe()

statistic,asin,len
str,str,f64
"""count""","""603029""",603029.0
"""null_count""","""0""",0.0
"""mean""",null,8.818705
"""std""",null,47.629373
"""min""","""0004457196""",1.0
"""25%""",null,1.0
"""50%""",null,2.0
"""75%""",null,6.0
"""max""","""BT00KRV6SY""",12759.0


In [85]:
df1 = df1.join(count1, on='asin')

df2 = df1.filter(pl.col('len') > 5)

In [89]:
df3 = df2.filter(pl.col('timestamp') > 1671321600000)

df3.height / df2.height, df3.height

(0.020380291200750838, 91028)

In [90]:
train = df2.filter(pl.col('timestamp') <= 1671321600000)
test = df2.filter(pl.col('timestamp') > 1671321600000)

In [91]:
uid = train.select('user_id')

test = test.join(uid, on='user_id', how='semi') 

In [92]:
test.height, train.height

(36741, 4375444)